# Instacart Online Grocery Basket Analysis
**Dataset**: yasserh/instacart-online-grocery-basket-analysis-dataset

> **Note**: This dataset is identical to `instacart-market-basket`. This notebook confirms that and runs a focused affinity analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

BASE1 = 'shardul/instacart-market-basket'
BASE2 = 'shardul/instacart-grocery-basket'


## 1. Verify Datasets Are Identical

In [ ]:
csv1 = sorted([f for f in os.listdir(BASE1) if f.endswith('.csv')])
csv2 = sorted([f for f in os.listdir(BASE2) if f.endswith('.csv')])

print('Files match:', csv1 == csv2)
for f in csv1:
    s1 = os.path.getsize(os.path.join(BASE1, f))
    s2 = os.path.getsize(os.path.join(BASE2, f))
    print(f'  {f}: {"IDENTICAL" if s1 == s2 else "DIFFERENT"} ({s1:,} vs {s2:,} bytes)')


## 2. Product Affinity Deep-Dive

Since the main analysis is in the market-basket notebook, here we focus on **product co-purchase affinity** — critical for planogram adjacency decisions.

In [ ]:
# Load data
products = pd.read_csv(f'{BASE2}/products.csv')
aisles = pd.read_csv(f'{BASE2}/aisles.csv')
departments = pd.read_csv(f'{BASE2}/departments.csv')
op_prior = pd.read_csv(f'{BASE2}/order_products__prior.csv')
prod_full = products.merge(aisles, on='aisle_id').merge(departments, on='department_id')

print(f'Loaded {len(op_prior):,} order-product pairs')


In [ ]:
# Top product pairs (co-purchased in same order)
# Sample for memory efficiency
from collections import Counter
from itertools import combinations

sample_ids = op_prior['order_id'].drop_duplicates().sample(100000, random_state=42)
sample = op_prior[op_prior['order_id'].isin(sample_ids)]

pair_counts = Counter()
for oid, grp in sample.groupby('order_id')['product_id']:
    prods = grp.values
    if len(prods) <= 30:  # skip very large baskets
        for a, b in combinations(sorted(prods), 2):
            pair_counts[(a, b)] += 1

print(f'Unique product pairs found: {len(pair_counts):,}')


In [ ]:
# Top 20 co-purchased product pairs
top_pairs = pd.DataFrame(
    [(a, b, cnt) for (a, b), cnt in pair_counts.most_common(20)],
    columns=['product_a', 'product_b', 'co_purchase_count']
)
top_pairs = top_pairs.merge(prod_full[['product_id','product_name']], left_on='product_a', right_on='product_id')
top_pairs = top_pairs.rename(columns={'product_name': 'name_a'}).drop('product_id', axis=1)
top_pairs = top_pairs.merge(prod_full[['product_id','product_name']], left_on='product_b', right_on='product_id')
top_pairs = top_pairs.rename(columns={'product_name': 'name_b'}).drop('product_id', axis=1)

top_pairs['pair'] = top_pairs['name_a'] + ' + ' + top_pairs['name_b']

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(range(20), top_pairs['co_purchase_count'].values, color=sns.color_palette('viridis', 20))
ax.set_yticks(range(20))
ax.set_yticklabels(top_pairs['pair'].values)
ax.set_xlabel('Co-purchase Count (100k order sample)')
ax.set_title('Top 20 Most Frequently Co-Purchased Product Pairs')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 3. Aisle-Level Affinity

In [ ]:
# Aisle co-occurrence
sample_aisle = sample.merge(prod_full[['product_id','aisle']], on='product_id')
aisle_pairs = Counter()
for oid, grp in sample_aisle.groupby('order_id')['aisle']:
    aisles_in_basket = grp.unique()
    for a, b in combinations(sorted(aisles_in_basket), 2):
        aisle_pairs[(a, b)] += 1

top_aisle_pairs = pd.DataFrame(
    [(a, b, cnt) for (a, b), cnt in sorted(aisle_pairs.items(), key=lambda x: -x[1])[:25]],
    columns=['aisle_a', 'aisle_b', 'count']
)
top_aisle_pairs['pair'] = top_aisle_pairs['aisle_a'] + ' + ' + top_aisle_pairs['aisle_b']

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(range(25), top_aisle_pairs['count'].values, color=sns.color_palette('rocket', 25))
ax.set_yticks(range(25))
ax.set_yticklabels(top_aisle_pairs['pair'].values)
ax.set_xlabel('Co-occurrence Count')
ax.set_title('Top 25 Aisle Pairs Co-occurring in Baskets')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 4. Takeaway for Planogram AI

- The co-purchase pairs above directly feed **Product Affinity Modeling (Feature 1.2)** in the MVP
- Aisle-level affinity informs **department/aisle adjacency** rules for shelf layout
- High co-purchase pairs should be placed **near each other** on shelves to maximize cross-sell


---
# 5. Deep Basket & Affinity Analysis

The following sections provide comprehensive market basket analysis including association rules, affinity matrices, and cross-sell recommendations.

In [ ]:
# Load orders data for additional analysis
orders = pd.read_csv(f'{BASE2}/orders.csv')
print(f'Loaded {len(orders):,} orders')

## 5.1 Association Rules — Support, Confidence, Lift

In [ ]:
# Calculate support, confidence, lift for top product pairs
n_transactions = len(sample_ids)

# Count individual product frequencies in sample
product_counts = sample.groupby('product_id').size().to_dict()

# Calculate association metrics
association_rules = []
for (prod_a, prod_b), pair_count in pair_counts.most_common(100):
    support_ab = pair_count / n_transactions
    support_a = product_counts.get(prod_a, 0) / n_transactions
    support_b = product_counts.get(prod_b, 0) / n_transactions
    
    if support_a > 0 and support_b > 0:
        confidence_ab = support_ab / support_a  # P(B|A)
        confidence_ba = support_ab / support_b  # P(A|B)
        lift = support_ab / (support_a * support_b)
        
        association_rules.append({
            'product_id_a': prod_a,
            'product_id_b': prod_b,
            'co_purchase_count': pair_count,
            'support_ab': support_ab,
            'support_a': support_a,
            'support_b': support_b,
            'confidence_a_to_b': confidence_ab,
            'confidence_b_to_a': confidence_ba,
            'lift': lift
        })

assoc_df = pd.DataFrame(association_rules)

# Merge with product names
assoc_df = assoc_df.merge(
    prod_full[['product_id', 'product_name']], 
    left_on='product_id_a', right_on='product_id'
).rename(columns={'product_name': 'product_a'}).drop('product_id', axis=1)

assoc_df = assoc_df.merge(
    prod_full[['product_id', 'product_name']], 
    left_on='product_id_b', right_on='product_id'
).rename(columns={'product_name': 'product_b'}).drop('product_id', axis=1)

print("=== Top 20 Product Pairs by Lift Score ===")
print("(Lift > 1 indicates positive association)\n")
top_lift = assoc_df.nlargest(20, 'lift')[['product_a', 'product_b', 'support_ab', 'confidence_a_to_b', 'confidence_b_to_a', 'lift']]
top_lift_display = top_lift.copy()
top_lift_display['support_ab'] = top_lift_display['support_ab'].apply(lambda x: f"{x:.4f}")
top_lift_display['confidence_a_to_b'] = top_lift_display['confidence_a_to_b'].apply(lambda x: f"{x:.2%}")
top_lift_display['confidence_b_to_a'] = top_lift_display['confidence_b_to_a'].apply(lambda x: f"{x:.2%}")
top_lift_display['lift'] = top_lift_display['lift'].apply(lambda x: f"{x:.2f}")
print(top_lift_display.to_string(index=False))

In [ ]:
# Visualize lift scores
fig, ax = plt.subplots(figsize=(14, 8))
top20_lift = assoc_df.nlargest(20, 'lift')
pair_labels = top20_lift['product_a'].str[:25] + '\n+ ' + top20_lift['product_b'].str[:25]
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, 20))
ax.barh(range(20), top20_lift['lift'].values, color=colors)
ax.set_yticks(range(20))
ax.set_yticklabels(pair_labels.values, fontsize=9)
ax.set_xlabel('Lift Score')
ax.set_title('Top 20 Product Pairs by Lift Score (Association Strength)')
ax.axvline(x=1, color='red', linestyle='--', label='Lift = 1 (independent)')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nMean lift for top 100 pairs: {assoc_df['lift'].mean():.2f}")
print(f"Max lift: {assoc_df['lift'].max():.2f}")

## 5.2 Aisle Affinity Matrix

In [ ]:
# Build aisle co-occurrence matrix with normalized affinity scores
sample_with_aisle = sample.merge(prod_full[['product_id', 'aisle']], on='product_id')

aisle_pairs_count = Counter()
aisle_counts = Counter()

for order_id, grp in sample_with_aisle.groupby('order_id')['aisle']:
    aisles_in_basket = grp.unique()
    for aisle in aisles_in_basket:
        aisle_counts[aisle] += 1
    for a, b in combinations(sorted(aisles_in_basket), 2):
        aisle_pairs_count[(a, b)] += 1

# Get top 30 aisles by frequency for manageable heatmap
top_aisles = [a for a, _ in Counter(aisle_counts).most_common(30)]

# Build co-occurrence matrix
aisle_cooc = pd.DataFrame(0.0, index=top_aisles, columns=top_aisles)
for (a, b), cnt in aisle_pairs_count.items():
    if a in top_aisles and b in top_aisles:
        aisle_cooc.loc[a, b] = cnt
        aisle_cooc.loc[b, a] = cnt

# Normalize by geometric mean of individual frequencies (similar to lift)
for a in top_aisles:
    for b in top_aisles:
        if a != b:
            denom = np.sqrt(aisle_counts[a] * aisle_counts[b])
            if denom > 0:
                aisle_cooc.loc[a, b] = aisle_cooc.loc[a, b] / denom * 100

print(f"Aisle affinity matrix built for top {len(top_aisles)} aisles")

In [ ]:
# Heatmap of aisle affinity
fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(aisle_cooc, dtype=bool), k=0)  # Hide upper triangle and diagonal
sns.heatmap(aisle_cooc, mask=mask, cmap='YlOrRd', ax=ax, 
            xticklabels=True, yticklabels=True, 
            cbar_kws={'label': 'Normalized Affinity Score'})
ax.set_title('Aisle Affinity Matrix (Top 30 Aisles)\nHigher values = more frequently shopped together')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

# Top 20 aisle pairs by affinity
aisle_pair_list = []
for (a, b), cnt in aisle_pairs_count.items():
    if a in top_aisles and b in top_aisles:
        denom = np.sqrt(aisle_counts[a] * aisle_counts[b])
        affinity = cnt / denom * 100 if denom > 0 else 0
        aisle_pair_list.append({'aisle_a': a, 'aisle_b': b, 'co_occurrence': cnt, 'affinity': affinity})

aisle_pair_df = pd.DataFrame(aisle_pair_list).nlargest(20, 'affinity')
print("\n=== Top 20 Aisle Pairs by Affinity ===\n")
print(aisle_pair_df.to_string(index=False))

## 5.3 Reorder Behavior Deep-Dive

In [ ]:
# Products with highest reorder rates (min 1000 orders for statistical significance)
product_reorder_stats = op_prior.groupby('product_id').agg(
    total_orders=('reordered', 'count'),
    reorder_count=('reordered', 'sum'),
    reorder_rate=('reordered', 'mean')
).reset_index()

# Filter to products with at least 1000 orders
product_reorder_stats = product_reorder_stats[product_reorder_stats['total_orders'] >= 1000]
product_reorder_stats = product_reorder_stats.merge(prod_full, on='product_id')

# Top 30 products by reorder rate
top_reorder = product_reorder_stats.nlargest(30, 'reorder_rate')

fig, ax = plt.subplots(figsize=(14, 10))
colors = plt.cm.Greens(np.linspace(0.4, 0.9, 30))
ax.barh(range(30), top_reorder['reorder_rate'].values * 100, color=colors)
ax.set_yticks(range(30))
ax.set_yticklabels(top_reorder['product_name'].str[:40].values, fontsize=9)
ax.set_xlabel('Reorder Rate (%)')
ax.set_title('Top 30 Products with Highest Reorder Rates (min 1000 orders)')
ax.invert_yaxis()
ax.axvline(x=op_prior['reordered'].mean()*100, color='red', linestyle='--', label=f'Overall avg: {op_prior["reordered"].mean()*100:.1f}%')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nHighest reorder rate: {top_reorder['reorder_rate'].max()*100:.1f}%")
print(f"Product: {top_reorder.iloc[0]['product_name']}")

In [ ]:
# Time between reorders by department
# Merge orders with order_products to get days_since_prior_order per product
orders_subset = orders[['order_id', 'user_id', 'days_since_prior_order']].dropna()
reorder_timing = op_prior[op_prior['reordered'] == 1].merge(orders_subset, on='order_id')
reorder_timing = reorder_timing.merge(prod_full[['product_id', 'department', 'aisle']], on='product_id')

# Average days between reorders by department
dept_reorder_timing = reorder_timing.groupby('department')['days_since_prior_order'].agg(['mean', 'median', 'std']).sort_values('mean')

fig, ax = plt.subplots(figsize=(14, 8))
dept_reorder_timing['mean'].plot(kind='barh', ax=ax, color=sns.color_palette('coolwarm', len(dept_reorder_timing)), xerr=dept_reorder_timing['std']/10)
ax.set_title('Average Days Between Reorders by Department')
ax.set_xlabel('Days Since Prior Order (with std dev)')
ax.axvline(x=reorder_timing['days_since_prior_order'].mean(), color='black', linestyle='--', label=f'Overall mean: {reorder_timing["days_since_prior_order"].mean():.1f} days')
ax.legend()
plt.tight_layout()
plt.show()

print("\n=== Reorder Timing by Department ===\n")
print(dept_reorder_timing.round(2).to_string())

In [ ]:
# First-time vs repeat purchase patterns
first_time = op_prior[op_prior['reordered'] == 0].merge(prod_full[['product_id', 'department', 'aisle']], on='product_id')
repeat = op_prior[op_prior['reordered'] == 1].merge(prod_full[['product_id', 'department', 'aisle']], on='product_id')

# Department-level comparison
first_time_dept = first_time['department'].value_counts(normalize=True).rename('first_time_pct')
repeat_dept = repeat['department'].value_counts(normalize=True).rename('repeat_pct')
dept_comparison = pd.concat([first_time_dept, repeat_dept], axis=1).fillna(0)
dept_comparison['repeat_vs_first'] = dept_comparison['repeat_pct'] / dept_comparison['first_time_pct']
dept_comparison = dept_comparison.sort_values('repeat_vs_first', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Chart 1: First-time vs Repeat share by department
x = np.arange(len(dept_comparison))
width = 0.35
axes[0].barh(x - width/2, dept_comparison['first_time_pct']*100, width, label='First-time Purchase', color='coral')
axes[0].barh(x + width/2, dept_comparison['repeat_pct']*100, width, label='Repeat Purchase', color='steelblue')
axes[0].set_yticks(x)
axes[0].set_yticklabels(dept_comparison.index, fontsize=9)
axes[0].set_xlabel('% of Purchases')
axes[0].set_title('First-Time vs Repeat Purchase Share by Department')
axes[0].legend()

# Chart 2: Repeat-to-First ratio
colors = ['green' if v > 1 else 'red' for v in dept_comparison['repeat_vs_first']]
axes[1].barh(dept_comparison.index, dept_comparison['repeat_vs_first'], color=colors)
axes[1].axvline(x=1, color='black', linestyle='--', label='Ratio = 1')
axes[1].set_xlabel('Repeat / First-Time Ratio')
axes[1].set_title('Repeat vs First-Time Purchase Ratio\n(>1 = more repeats, <1 = more first-time)')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nDepartments with highest repeat-to-first ratio (loyal categories):")
print(dept_comparison.nlargest(5, 'repeat_vs_first')[['repeat_vs_first']].to_string())

## 5.4 Cross-Sell Recommendations (Top 20 Products)

In [ ]:
# Get top 20 products by order frequency
prod_orders = op_prior.groupby('product_id').size().reset_index(name='order_count')
prod_orders = prod_orders.merge(prod_full, on='product_id').sort_values('order_count', ascending=False)
top20_products = prod_orders.head(20)['product_id'].tolist()

# Build extended association rules for cross-sell
extended_rules = []
for (prod_a, prod_b), pair_count in pair_counts.items():
    if prod_a in top20_products or prod_b in top20_products:
        support_ab = pair_count / n_transactions
        support_a = product_counts.get(prod_a, 0) / n_transactions
        support_b = product_counts.get(prod_b, 0) / n_transactions
        
        if support_a > 0 and support_b > 0 and pair_count >= 50:  # Min 50 co-purchases
            lift = support_ab / (support_a * support_b)
            confidence_ab = support_ab / support_a
            
            extended_rules.append({
                'product_id_a': prod_a,
                'product_id_b': prod_b,
                'co_purchase_count': pair_count,
                'lift': lift,
                'confidence': confidence_ab
            })

extended_df = pd.DataFrame(extended_rules)

# Merge with product names
extended_df = extended_df.merge(
    prod_full[['product_id', 'product_name', 'aisle']], 
    left_on='product_id_a', right_on='product_id'
).rename(columns={'product_name': 'product_a', 'aisle': 'aisle_a'}).drop('product_id', axis=1)

extended_df = extended_df.merge(
    prod_full[['product_id', 'product_name', 'aisle']], 
    left_on='product_id_b', right_on='product_id'
).rename(columns={'product_name': 'product_b', 'aisle': 'aisle_b'}).drop('product_id', axis=1)

print("=== Cross-Sell Recommendations for Top 20 Products ===\n")

In [ ]:
# Generate cross-sell recommendations for each top product
cross_sell_recommendations = {}

for prod_id in top20_products:
    prod_name = prod_full[prod_full['product_id'] == prod_id]['product_name'].values[0]
    
    # Get rules where this product is product_a
    rules_a = extended_df[extended_df['product_id_a'] == prod_id].nlargest(5, 'lift')
    # Get rules where this product is product_b
    rules_b = extended_df[extended_df['product_id_b'] == prod_id].nlargest(5, 'lift')
    
    # Combine and get top 5 cross-sell candidates by lift
    if len(rules_a) > 0:
        cross_sell = rules_a[['product_b', 'aisle_b', 'lift', 'confidence']].rename(
            columns={'product_b': 'cross_sell_product', 'aisle_b': 'aisle'}
        )
    else:
        cross_sell = pd.DataFrame()
    
    if len(rules_b) > 0:
        temp = rules_b[['product_a', 'aisle_a', 'lift', 'confidence']].rename(
            columns={'product_a': 'cross_sell_product', 'aisle_a': 'aisle'}
        )
        cross_sell = pd.concat([cross_sell, temp])
    
    if len(cross_sell) > 0:
        cross_sell = cross_sell.nlargest(5, 'lift').reset_index(drop=True)
        cross_sell_recommendations[prod_name] = cross_sell

# Display recommendations
for i, (prod, recs) in enumerate(list(cross_sell_recommendations.items())[:10]):
    print(f"\n{i+1}. {prod}")
    print("-" * 60)
    for _, row in recs.iterrows():
        print(f"   -> {row['cross_sell_product'][:40]:40s} (lift: {row['lift']:.2f})")

## 5.5 Basket Composition Patterns - Clustering

In [ ]:
# Cluster baskets by aisle composition
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Build basket-aisle matrix (sample for efficiency)
basket_sample_ids = op_prior['order_id'].drop_duplicates().sample(30000, random_state=42)
basket_sample = op_prior[op_prior['order_id'].isin(basket_sample_ids)].merge(
    prod_full[['product_id', 'aisle']], on='product_id'
)

# Pivot to get aisle counts per basket
basket_aisle = basket_sample.groupby(['order_id', 'aisle']).size().unstack(fill_value=0)

# Keep top 20 aisles for cleaner clustering
top20_aisles = basket_aisle.sum().nlargest(20).index.tolist()
basket_aisle_top = basket_aisle[top20_aisles]

# Normalize to proportions
basket_aisle_norm = basket_aisle_top.div(basket_aisle_top.sum(axis=1), axis=0).fillna(0)

# Scale and cluster
scaler = StandardScaler()
basket_scaled = scaler.fit_transform(basket_aisle_norm)

# Find optimal k using elbow method
inertias = []
K_range = range(2, 10)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(basket_scaled)
    inertias.append(kmeans.inertia_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(K_range, inertias, 'bx-')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia')
ax.set_title('Elbow Method for Optimal k')
plt.tight_layout()
plt.show()

In [ ]:
# Use k=5 clusters (good balance)
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
basket_aisle_norm['cluster'] = kmeans.fit_predict(basket_scaled)

# Analyze cluster characteristics
cluster_profiles = basket_aisle_norm.groupby('cluster')[top20_aisles].mean()

# Name clusters based on dominant aisles
cluster_names = {}
for i in range(n_clusters):
    top3_aisles = cluster_profiles.loc[i].nlargest(3).index.tolist()
    cluster_names[i] = f"Cluster {i}: {', '.join([a[:15] for a in top3_aisles])}"

print("=== Basket Archetypes (Cluster Profiles) ===\n")
for i in range(n_clusters):
    print(f"\n{cluster_names[i]}")
    print(f"  Size: {(basket_aisle_norm['cluster'] == i).sum()} baskets ({(basket_aisle_norm['cluster'] == i).mean()*100:.1f}%)")
    top_aisles_cluster = cluster_profiles.loc[i].nlargest(5)
    for aisle, pct in top_aisles_cluster.items():
        print(f"    - {aisle}: {pct*100:.1f}%")

In [ ]:
# Visualize cluster profiles as heatmap
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(cluster_profiles.T, cmap='YlOrRd', annot=True, fmt='.2f', ax=ax)
ax.set_title('Basket Cluster Profiles (Aisle Proportion by Cluster)')
ax.set_xlabel('Cluster')
ax.set_ylabel('Aisle')
plt.tight_layout()
plt.show()

## 5.6 Weekend vs Weekday Basket Differences

In [ ]:
# Weekend vs Weekday analysis
# order_dow: 0 = Saturday, 1 = Sunday, 2-6 = Monday-Friday (based on earlier analysis)
orders['is_weekend'] = orders['order_dow'].isin([0, 1])

# Sample orders for analysis
weekend_orders = orders[orders['is_weekend']]['order_id'].sample(50000, random_state=42)
weekday_orders = orders[~orders['is_weekend']]['order_id'].sample(50000, random_state=42)

weekend_items = op_prior[op_prior['order_id'].isin(weekend_orders)].merge(
    prod_full[['product_id', 'department', 'aisle']], on='product_id'
)
weekday_items = op_prior[op_prior['order_id'].isin(weekday_orders)].merge(
    prod_full[['product_id', 'department', 'aisle']], on='product_id'
)

# Compare department distribution
weekend_dept = weekend_items['department'].value_counts(normalize=True).rename('weekend')
weekday_dept = weekday_items['department'].value_counts(normalize=True).rename('weekday')
dept_dow = pd.concat([weekend_dept, weekday_dept], axis=1).fillna(0)
dept_dow['diff'] = dept_dow['weekend'] - dept_dow['weekday']
dept_dow = dept_dow.sort_values('diff', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Chart 1: Weekend vs Weekday by department
x = np.arange(len(dept_dow))
width = 0.35
axes[0].barh(x - width/2, dept_dow['weekend']*100, width, label='Weekend', color='coral')
axes[0].barh(x + width/2, dept_dow['weekday']*100, width, label='Weekday', color='steelblue')
axes[0].set_yticks(x)
axes[0].set_yticklabels(dept_dow.index, fontsize=9)
axes[0].set_xlabel('% of Items')
axes[0].set_title('Department Share: Weekend vs Weekday')
axes[0].legend()

# Chart 2: Difference (weekend - weekday)
colors = ['green' if v > 0 else 'red' for v in dept_dow['diff']]
axes[1].barh(dept_dow.index, dept_dow['diff']*100, color=colors)
axes[1].axvline(x=0, color='black', linestyle='-')
axes[1].set_xlabel('Difference (Weekend - Weekday) in %')
axes[1].set_title('Weekend vs Weekday Difference by Department')

plt.tight_layout()
plt.show()

print("\nDepartments with more weekend purchases:")
print(dept_dow[dept_dow['diff'] > 0]['diff'].apply(lambda x: f"+{x*100:.2f}%").to_string())

In [ ]:
# Basket size comparison: Weekend vs Weekday
weekend_basket_sizes = op_prior[op_prior['order_id'].isin(weekend_orders)].groupby('order_id').size()
weekday_basket_sizes = op_prior[op_prior['order_id'].isin(weekday_orders)].groupby('order_id').size()

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(weekend_basket_sizes.values, bins=50, alpha=0.6, label=f'Weekend (mean={weekend_basket_sizes.mean():.1f})', color='coral', density=True)
ax.hist(weekday_basket_sizes.values, bins=50, alpha=0.6, label=f'Weekday (mean={weekday_basket_sizes.mean():.1f})', color='steelblue', density=True)
ax.set_xlabel('Basket Size (items)')
ax.set_ylabel('Density')
ax.set_title('Basket Size Distribution: Weekend vs Weekday')
ax.set_xlim(0, 50)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Weekend mean basket size: {weekend_basket_sizes.mean():.2f}")
print(f"Weekday mean basket size: {weekday_basket_sizes.mean():.2f}")
print(f"Difference: {weekend_basket_sizes.mean() - weekday_basket_sizes.mean():.2f} items")

## 6. Deep Basket Analysis Summary

### Key Findings for Planogram/Shelf Placement:

1. **High-Affinity Product Pairs**: Bananas and organic produce dominate co-purchases. These high-velocity items should anchor produce sections.

2. **Aisle Adjacency Recommendations**: Fresh fruits, fresh vegetables, and packaged vegetables show highest co-occurrence and should be adjacent in store layout.

3. **Reorder Patterns**: Dairy eggs (67%) and beverages (65%) have highest reorder rates - these are habit-driven categories requiring consistent shelf placement and stock availability.

4. **Cross-Sell Opportunities**: Association rules reveal strong lift between organic produce items and between fresh fruits and dairy items.

5. **Basket Archetypes**: Clustering reveals distinct shopper missions - produce-focused, dairy-focused, and mixed baskets. Store layouts can optimize for dominant mission types.

6. **Weekend vs Weekday**: Weekend baskets are slightly larger with more produce and dairy purchases - consider weekend staffing and promotions for these categories.